# Abstract

- Goal: Train BERT on Sentiment Analysis task and reach good accuracy score

- Dataset: [IMDB Dataset of 50K Movie Reviews (Kaggle)](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)

- Project Details:

    I'm using standart architecture for LSTM Classifiers (Embedding -> LSTM -> average pooling (optional) -> FC), model made with pytorch. This model trained to analyze sentiment on IMDB Movie Reviews dataset.

- Best result: 0.8938 (Accuracy Score of DIY LSTM Classifier)

- Sections:
    - [Imports](#Imports)
    - [Dataset](#Dataset)
    - [Modeling](#Modeling)
    - [Prediction](#Prediction)
        - [Setup Form](#Setup-Form)
        - [Prediction Form](#Prediction-Form)

## Problems & Solutions
- Problem 1: Model is underfitted or model has high bias

    **Symptoms:**

    - Model's train/validation loss and accuracy changing very slowly

    **Root Cause:**

    - Big size of model
    - Small learing rate
    - Small batch size

    **Solution:**

    - Reduce model size LSTM's num_layers 8 -> 2, Hidden and Embedding dim 256 -> 64
    - Change model's learning rate 3e-4 -> 1e-3 and batch size 32 -> 64

# Download & Install Dependencies

In [ ]:
# Install modules (if needed)
!pip install torch ipywidgets ipython ipython huggingface-hub pandas tensorflow tqdm

In [ ]:
# Dataset Downloading
!mkdir data
!curl -L -o ./data/dataset.zip https://www.kaggle.com/api/v1/datasets/download/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
!unzip ./data/dataset.zip -d ./data
!rm ./data/dataset.zip
!mv ./data/IMDB\ Dataset.csv ./data/imdb_dataset.csv

In [1]:
# Model Downloading
!mkdir models
from huggingface_hub import snapshot_download

PROJECT_NAME = "SentimentAnalysisIMDB50K"
MODEL_FOLDER = "lstm"
repo_id = f"jsonmen/{PROJECT_NAME}"

snapshot_download(
    repo_id=repo_id,
    local_dir="./models",
    allow_patterns=[f"{MODEL_FOLDER}/*"],
    token=False  # No token needed for public repos
)
print(f"Downloaded models folder from {repo_id} to ./models")

mkdir: cannot create directory ‘models’: File exists


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

tokenizer.pkl:   0%|          | 0.00/5.41M [00:00<?, ?B/s]

lstm_model.pt:   0%|          | 0.00/2.83M [00:00<?, ?B/s]

Downloaded models folder from jsonmen/SentimentAnalysisIMDB50K to ./models


In [ ]:
# For colab users (local library files download)
!curl -L -o ./setup_text_preprocessing.py https://raw.githubusercontent.com/jsonmen/bias-and-variance/refs/heads/main/SentimentAnalysis/setup_text_preprocessing.py
!curl -L -o ./text_preprocessing.py https://raw.githubusercontent.com/jsonmen/bias-and-variance/refs/heads/main/SentimentAnalysis/text_preprocessing.py

# Imports

In [2]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from text_preprocessing import text_preprocessing
from torch import nn
from tqdm import tqdm
import pickle
import ipywidgets as widgets
from IPython.display import clear_output, display
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Dataset

In [2]:
sentiment2label = {"positive": 1, "negative": 0}
label2sentiment = {1: "positive", 0: "negative"}

In [3]:
class IMDBDataset(Dataset):
    def __init__(self, dataset_path, tokenizer, sequence_maxlen=600):
        # cache all transfomations because dataset isn't large
        self.dataset = pd.read_csv(dataset_path)
        self.sequences = tokenizer.texts_to_sequences(self.dataset["review"])
        self.padded_sequences = pad_sequences(self.sequences, maxlen=sequence_maxlen, padding='post')
        self.labels = self.dataset["sentiment"].map(sentiment2label)
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        return torch.tensor(self.padded_sequences[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float32)

# Modeling

A few words about LSTM:

**LSTM** stands for **Long Short-Term Memory**, a type of recurrent neural network (RNN) designed to model and learn from sequential data, particularly effective for tasks involving long-term dependencies.

LSTMs consist of multiple **LSTM units organized in layers**. Each unit contains a cell that maintains information over time and three key gates:

- **Input gate**: Controls what new information is added to the cell.
- **Forget gate**: Decides what information to discard from the cell.
- **Output gate**: Determines what information is output to the next layer or time step.

For tasks like sentiment analysis (a form of classification), the LSTM's outputs are fed into additional classification layers to predict outcomes, such as positive or negative sentiment.

Here’s an image illustrating how an **LSTM unit** works also that image show difference between **RNN** and **LSTM**:

<img src="https://www.researchgate.net/publication/332662013/figure/fig3/AS:751758288637957@1556244551951/RNN-and-LSTM-comparison-chart.jpg" alt="LSTM unit diagram" width="500">

In [4]:
class LSTMClassificationModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_layers, hidden_dim, output_dim=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, X):
        embedded = self.embedding(X) 
        output, (hidden, cell) = self.lstm(embedded)

        # Average pool across time steps
        pooled = output.mean(dim=1)
        out = self.fc(pooled)

        return out

In [5]:
# Model Hyper Parameters
VOCAB_SIZE = 10_000
EMBEDDING_DIM = 64
N_LAYERS = 2
HIDDEN_DIM = 64
OUTPUT_DIM = 1
SEQ_MAXLEN = 600
NUM_WORKERS = 2
LR = 1e-3
BATCH_SIZE = 64
EPOCH = 10
DATASET_PATH = "./data/imdb_dataset.csv"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
# create tokenizer
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")

pd_dataset = pd.read_csv(DATASET_PATH)
tokenizer.fit_on_texts(pd_dataset["review"])

In [7]:
model = LSTMClassificationModel(VOCAB_SIZE, EMBEDDING_DIM, N_LAYERS, HIDDEN_DIM, OUTPUT_DIM)
model = model.to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()

dataset = IMDBDataset(DATASET_PATH, tokenizer, SEQ_MAXLEN)
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

In [8]:
def binary_accuracy(y_pred, y_true):
    """
    Compute accuracy for binary classification.
    """
    # sigmoid → probabilities
    probs = torch.sigmoid(y_pred)
    preds = torch.round(probs)
    correct = (preds == y_true).float()
    acc = correct.sum() / len(correct)
    return acc

def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    """
    Train model for one epoch.
    """
    model.train()
    running_loss = 0.0
    running_acc = 0.0

    progress_bar = tqdm(dataloader, desc="Train", leave=False)

    for inputs, targets in progress_bar:
        inputs = inputs.to(device)
        targets = targets.to(device).float().unsqueeze(1)  # Shape: (batch, 1)

        optimizer.zero_grad()
        outputs = model(inputs)

        loss = loss_fn(outputs, targets)
        acc = binary_accuracy(outputs, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_acc += acc.item() * inputs.size(0)

        avg_loss = running_loss / ((progress_bar.n + 1) * inputs.size(0))
        avg_acc = running_acc / ((progress_bar.n + 1) * inputs.size(0))
        progress_bar.set_postfix(loss=avg_loss, acc=avg_acc)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc


def evaluate(model, dataloader, loss_fn, device):
    """
    Evaluate model.
    """
    model.eval()
    running_loss = 0.0
    running_acc = 0.0

    progress_bar = tqdm(dataloader, desc="Val", leave=False)

    with torch.no_grad():
        for inputs, targets in progress_bar:
            inputs = inputs.to(device)
            targets = targets.to(device).float().unsqueeze(1)

            outputs = model(inputs)
            loss = loss_fn(outputs, targets)
            acc = binary_accuracy(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            running_acc += acc.item() * inputs.size(0)

            avg_loss = running_loss / ((progress_bar.n + 1) * inputs.size(0))
            avg_acc = running_acc / ((progress_bar.n + 1) * inputs.size(0))
            progress_bar.set_postfix(loss=avg_loss, acc=avg_acc)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc


def train_loop(model, train_loader, val_loader, optimizer, loss_fn, device, epochs):
    """
    Run full training loop.
    """
    # Example usage:
    # train_loop(
    #     model,
    #     train_loader,
    #     val_loader,
    #     optimizer,
    #     loss_fn,
    #     DEVICE,
    #     EPOCH
    # )
    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, loss_fn, device
        )

        val_loss, val_acc = evaluate(
            model, val_loader, loss_fn, device
        )

        print(
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
            f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )


In [9]:
train_loop(model, train_loader, val_loader, optimizer, loss_fn, DEVICE, EPOCH)


Epoch 1/10


Train Loss: 0.5360 | Train Acc: 0.7176 | Val Loss: 0.3954 | Val Acc: 0.8276

Epoch 2/10


Train Loss: 0.3281 | Train Acc: 0.8629 | Val Loss: 0.3194 | Val Acc: 0.8700

Epoch 3/10


Train Loss: 0.2467 | Train Acc: 0.9028 | Val Loss: 0.2954 | Val Acc: 0.8848

Epoch 4/10


Train Loss: 0.2009 | Train Acc: 0.9236 | Val Loss: 0.2845 | Val Acc: 0.8915

Epoch 5/10


...

Train Loss: 0.1472 | Train Acc: 0.9451 | Val Loss: 0.3247 | Val Acc: 0.8852

Epoch 7/10


Train Loss: 0.1270 | Train Acc: 0.9540 | Val Loss: 0.3139 | Val Acc: 0.8937

Epoch 8/10


Train Loss: 0.1057 | Train Acc: 0.9628 | Val Loss: 0.3615 | Val Acc: 0.8945

Epoch 9/10


Train Loss: 0.0937 | Train Acc: 0.9667 | Val Loss: 0.3954 | Val Acc: 0.8939

Epoch 10/10


Train Loss: 0.0856 | Train Acc: 0.9696 | Val Loss: 0.3979 | Val Acc: 0.8938


In [10]:
torch.save(model.state_dict(), "./models/lstm/lstm_model.pt")
with open("./models/lstm/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Prediction

In [11]:
trained_model = LSTMClassificationModel(VOCAB_SIZE, EMBEDDING_DIM, N_LAYERS, HIDDEN_DIM, OUTPUT_DIM)
trained_model.load_state_dict(torch.load("./models/lstm/lstm_model.pt", map_location=torch.device('cpu')))
trained_tokenizer = None
with open("./models/lstm/tokenizer.pkl", "rb") as f:
    trained_tokenizer = pickle.load(f)

## Setup Form

In [12]:
text_field = widgets.Text(
    value='',
    placeholder='Type something',
    description='Text:',
    disabled=False   
)

submit_button = widgets.Button(description="Analyze Sentiment")
output = widgets.Output()

def predict_sentiment_lstm(text):
    seq = trained_tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=SEQ_MAXLEN, padding='post')
    tensor = torch.tensor(padded, dtype=torch.long)
    
    with torch.no_grad():
        logits = trained_model(tensor)
        probs = torch.sigmoid(logits)
        pred_class = torch.round(probs)[0, 0].item()
    sentiment_word = label2sentiment[pred_class]
    return sentiment_word

def form_fn(b):
    text = text_field.value
    sentiment_word = predict_sentiment_lstm(text)
    with output:
        clear_output()
        print(f"Your text has a {sentiment_word} sentiment")

submit_button.on_click(form_fn)

form = widgets.VBox([text_field, submit_button, output])

## Prediction Form

In [13]:
display(form)